In [ ]:
from dfg_ms_plexus.training_setup import *

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
# root = Path('F:/DATA/dfg_plexus/')
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')

# df = pd.read_csv(root / "radiomics_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / 'radiomics_and_cnn_features___combined.csv', delimiter=';')
# df = pd.read_csv(root / "radiomics_features___combined_SA___normalized.csv", delimiter=";")
df = pd.read_csv(root / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

#ft = df.drop(columns=['label', 'patID'])
ft = df.drop(columns=['label', 'patID', 'DWI (0no,1yes)', 'LesionVolume', 'DiseaseDuration', 'EDSS'])
label = df['label'].astype(int)
label, class_mapping = get_labels_hc_cis_ms(label)

print(f"{df.shape=}")
print(f"{ft.shape=}")
print(f"{label.shape=}")
print(f"{label.unique()=}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(ft, label, test_size=0.3, random_state=42, stratify=label)
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values
print(f"{X_train.shape=}")
print(f"{type(X_train)=}")

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler

var_threshold = VarianceThreshold(0.001)
X_train_pruned = var_threshold.fit_transform(X_train)
X_test_pruned = var_threshold.fit_transform(X_test)

# 1. Scale the data (Deep learning models prefer scaled data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pruned)
X_test_scaled = scaler.transform(X_test_pruned)

# 2. Select the top features (e.g., top 20) to reduce noise
selector = SelectKBest(score_func=f_classif, k=20)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# 3. Apply SMOTE ONLY to the training data to boost Class 1
#smote = SMOTE(random_state=42)
#X_train_resampled, y_train_resampled = smote.fit_resample(X_train_selected, y_train)

print(f"Original training shape: {X_train.shape}")
# print(f"Resampled training shape: {X_train_resampled.shape}")

#X_train = X_train_resampled
#y_train = y_train_resampled

X_train, X_test = X_train_scaled, X_test_scaled
# X_train, X_test = X_train_selected, X_test_selected

In [ ]:
import os
import random

import torch

from tabpfn import TabPFNClassifier

training_seeds = [0, 1, 2, 3, 4]
TABPFN_DEVICE = "cuda"

all_train_reports = []
all_test_reports = []
all_train_cms = []
all_test_cms = []
all_train_balanced_accuracy = []
all_test_balanced_accuracy = []
all_train_macro_f1 = []
all_test_macro_f1 = []
all_test_probas = []

labels_sorted = np.sort(np.unique(label))

# Ensure target_names follow the same order as labels_sorted.
# get_labels_hc_cis_ms usually returns a mapping like {"hc": 0, "CIS": 1, "MS": 2}.
if isinstance(class_mapping, dict) and all(np.issubdtype(type(v), np.integer) for v in class_mapping.values()):
    class_names = [
        name
        for name, _ in sorted(class_mapping.items(), key=lambda item: item[1])
    ]
else:
    class_names = list(class_mapping.keys())

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def report_to_df(report_dict):
    df_report = pd.DataFrame(report_dict).T
    return df_report[["precision", "recall", "f1-score", "support"]]


def to_numpy(X):
    if hasattr(X, "values"):
        return X.values
    return np.asarray(X)


def to_numpy_int(y):
    if hasattr(y, "values"):
        return y.values.astype(int)
    return np.asarray(y).astype(int)


def make_tabpfn_classifier(seed, device=TABPFN_DEVICE):
    """Create TabPFNClassifier while staying compatible across TabPFN versions."""
    return TabPFNClassifier(
        device=device,
        random_state=seed,
        balance_probabilities=True,
        eval_metric="balanced_accuracy",
        tuning_config={
            "calibrate_temperature": True,
            "tune_decision_thresholds": True,
        },
    )


In [ ]:
for seed in training_seeds:
    print(f"\n========== Training seed {seed} ==========")

    seed_everything(seed)

    clf = make_tabpfn_classifier(seed=seed, device=TABPFN_DEVICE)
    clf.fit(X_train, y_train)

    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)

    train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
    test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    train_macro_f1 = f1_score(y_train, y_train_pred, average="macro")
    test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

    print(f"Train balanced accuracy: {train_bal_acc:.3f}")
    print(f"Test balanced accuracy:  {test_bal_acc:.3f}")
    print(f"Train macro F1:          {train_macro_f1:.3f}")
    print(f"Test macro F1:           {test_macro_f1:.3f}")

    train_report = classification_report(
        y_train,
        y_train_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    test_report = classification_report(
        y_test,
        y_test_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    all_train_reports.append(report_to_df(train_report))
    all_test_reports.append(report_to_df(test_report))

    all_train_cms.append(
        confusion_matrix(y_train, y_train_pred, labels=labels_sorted)
    )
    all_test_cms.append(
        confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
    )

    all_train_balanced_accuracy.append(train_bal_acc)
    all_test_balanced_accuracy.append(test_bal_acc)
    all_train_macro_f1.append(train_macro_f1)
    all_test_macro_f1.append(test_macro_f1)
    all_test_probas.append(y_test_proba)

In [ ]:
tabpfn_seed_summary = pd.DataFrame({
    "training_seed": training_seeds,
    "train_balanced_accuracy": all_train_balanced_accuracy,
    "test_balanced_accuracy": all_test_balanced_accuracy,
    "train_macro_f1": all_train_macro_f1,
    "test_macro_f1": all_test_macro_f1,
})

display(tabpfn_seed_summary)

summary_mean_std = tabpfn_seed_summary.drop(columns="training_seed").agg(["mean", "std"]).T
display(summary_mean_std)

In [ ]:
train_summary = aggregate_reports(all_train_reports)
test_summary = aggregate_reports(all_test_reports)

print("\n--- Train Set: Mean ± Std over training seeds ---")
display(train_summary)

print("\n--- Test Set: Mean ± Std over training seeds ---")
display(test_summary)


In [ ]:
train_cms = np.asarray(all_train_cms)
test_cms = np.asarray(all_test_cms)

train_cms_norm = np.asarray([normalize_cm(cm) for cm in train_cms])
test_cms_norm = np.asarray([normalize_cm(cm) for cm in test_cms])

train_cm_mean = train_cms_norm.mean(axis=0)
train_cm_std = train_cms_norm.std(axis=0)

test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    train_cm_mean,
    train_cm_std,
    "Trainset Confusion Matrix: Mean ± Std over training seeds",
    class_names,
)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ± Std over training seeds",
    class_names,
)